<a href="https://colab.research.google.com/github/pranavvup-byte/ml-pratical-week/blob/main/pratical_week_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# --------------------------------------------------
# 1. Load dataset
# --------------------------------------------------

df = pd.read_csv("/content/placement_predict_50k_adjusted (1).csv")

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())


# --------------------------------------------------
# 2. Select features for clustering
# --------------------------------------------------

numeric_cols = [
    "CGPA",
    "AttendancePercent",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore"
]

categorical_cols = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "ExtraCurricular"
]


# --------------------------------------------------
# 3. Handle missing numerical values
# --------------------------------------------------

imputer = SimpleImputer(strategy="median")

num_df = pd.DataFrame(
    imputer.fit_transform(df[numeric_cols]),
    columns=numeric_cols
)


# --------------------------------------------------
# 4. Encode categorical variables
# --------------------------------------------------

cat_df = pd.get_dummies(
    df[categorical_cols],
    drop_first=True
)


# --------------------------------------------------
# 5. Combine features
# --------------------------------------------------

feature_df = pd.concat(
    [num_df, cat_df],
    axis=1
)


# --------------------------------------------------
# 6. Standardize features
# --------------------------------------------------

scaler = StandardScaler()

X = scaler.fit_transform(feature_df)

print("\nClustering features:", X.shape[1])


# --------------------------------------------------
# 7. Find best K using Silhouette Score
# --------------------------------------------------

k_values = range(2, 9)

inertias = []
silhouette_scores = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        n_init=10,
        random_state=42
    )

    labels = model.fit_predict(X)

    inertias.append(model.inertia_)
    silhouette_scores.append(
        silhouette_score(X, labels)
    )


# --------------------------------------------------
# 8. Plot Elbow and Silhouette
# --------------------------------------------------

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(
    list(k_values),
    inertias,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")


plt.subplot(1, 2, 2)

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")


plt.tight_layout()
plt.show()


# --------------------------------------------------
# 9. Select best K
# --------------------------------------------------

best_k = list(k_values)[
    np.argmax(silhouette_scores)
]

print("\nBest K:", best_k)


# --------------------------------------------------
# 10. Train final K-Means
# --------------------------------------------------

kmeans = KMeans(
    n_clusters=best_k,
    n_init=10,
    random_state=42
)

cluster_labels = kmeans.fit_predict(X)

print("\nK-Means Silhouette Score:",
      round(silhouette_score(X, cluster_labels), 4))


# --------------------------------------------------
# 11. Cluster sizes
# --------------------------------------------------

print("\nCluster sizes:")

print(
    pd.Series(cluster_labels)
    .value_counts()
    .sort_index()
)


# --------------------------------------------------
# 12. Analyze placement rate AFTER clustering
# --------------------------------------------------

df["KMeansCluster"] = cluster_labels

cluster_summary = df.groupby("KMeansCluster").agg(
    Students=("PlacementStatus", "size"),
    PlacementRate=("PlacementStatus", "mean"),
    AverageCGPA=("CGPA", "mean"),
    AverageInternships=("Internships", "mean"),
    AverageProjects=("Projects", "mean"),
    AverageAptitude=("AptitudeTestScore", "mean"),
    AverageCoding=("CodingTestScore", "mean")
)

print("\nCluster Profiles:")
print(cluster_summary.round(3))


# --------------------------------------------------
# 13. PCA visualization
# --------------------------------------------------

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X)

plt.figure(figsize=(8, 6))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=cluster_labels,
    s=8,
    alpha=0.6
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Student Clusters")

plt.show()


# --------------------------------------------------
# 14. Save results
# --------------------------------------------------

cluster_summary.to_csv(
    "cluster_profiles.csv"
)

df.to_csv(
    "placement_predict_with_clusters.csv",
    index=False
)

print("\nFiles saved:")
print("cluster_profiles.csv")
print("placement_predict_with_clusters.csv")

Dataset shape: (50000, 21)

First 5 rows:
   Gender       City CollegeTier      Stream Specialisation Hostel  \
0  Female      Delhi       Tier3          IT    DataScience    Yes   
1    Male    Chennai       Tier2         ECE             AI    Yes   
2  Female  Hyderabad       Tier3         ECE     Networking     No   
3  Female     Jaipur       Tier3         ECE       Embedded     No   
4    Male  Ahmedabad       Tier3  Mechanical    DataScience     No   

  HistoryOfBacklogs  CGPA  AttendancePercent  Internships  ...  Workshops  \
0               Yes  6.63               68.3            2  ...        0.0   
1                No  6.40               71.0            1  ...        0.0   
2                No  7.73               75.1            1  ...        2.0   
3                No  9.73               99.2            4  ...        5.0   
4                No  9.01               99.6            2  ...        NaN   

   Certifications  Publications  AptitudeTestScore  SoftSkillsRating  \
0 